In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torchvision.datasets import CIFAR10
from torchvision.transforms.functional import to_tensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch
from torch.optim import AdamW

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
import numpy as np


# 1. Convert Numpy arrays to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32)


In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)



In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print("Batch X shape:", X_batch.shape)
print("Batch y shape:", y_batch.shape)



In [ ]:
# 5. Display sample images
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    img = X_batch[i].permute(1, 2, 0)
    plt.imshow(img)
    plt.title(f"Age: {y_batch[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:

class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim):
        super(NN4Layer, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)

        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        self.layer4 = nn.Linear(hidden_dim, 1)

        # activation
        self.relu = nn.ReLU()

    def forward(self, x):
        # flatten image
        x = x.view(x.size(0), -1)

        # layer 1
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        # layer 2
        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        # layer 3
        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        # output layer
        output = self.layer4(a3)

        return output

In [ ]:

def train_one_epoch(model, optimizer, criterion, train_loader, device):
      # Set the model to training mode

    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1, 1).to(device)  # FIX shape

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(train_loader)


In [ ]:
def validate(model, criterion, test_loader, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.view(-1, 1).to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

    return running_loss / len(test_loader)  #



In [ ]:
# Task 4: Define device, model, loss, optimizer:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = X_train.shape[1] * X_train.shape[2] * X_train.shape[3]
hidden_dim = 256
output_dim = 10           # CIFAR-10 classes

model = NN4Layer(input_dim, hidden_dim).to(device)

print("Model Architecture:\n")
print(model)

num_epochs = 20
learning_rate = 0.001

criterion = nn.MSELoss()   #because it is regrission not classification
optimizer = AdamW(model.parameters(),learning_rate)


In [ ]:
# Task 5: Start training for 20 epochs:

train_losses = []
val_losses = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
    val_loss = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("Training Complete!")



In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here


# Show some test images with predicted age vs actual age

model.eval()
with torch.no_grad():
    X_vis, y_vis = next(iter(test_loader)) # take one batch
    X_vis = X_vis.to(device)

    preds = model(X_vis).cpu().numpy().flatten() # get model predictions

# plot examples
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    # change image shape to h, w, c for plotting
    img = X_vis[i].cpu().permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"prediciotn: {preds[i]:.1f} | actual: {y_vis[i].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()
